# 🧭 Citadel Governance Hub — Backend & Access Contract Tests

Test **any published LLM backend or access contract by name**. Set `TARGET` and run all cells: the notebook
discovers everything else from the deployed gateway.

| `TARGET` can be | Example | What is called |
|---|---|---|
| an access contract (file name in `environments/<ENV>/access-contracts/`) | `team-a-chatbot` | every product of the contract: each allowed model and alias on every LLM API the product grants, its MCP servers, plus guardrail checks |
| an APIM product | `LLM-TeamA-Chatbot-DEV` | the same, for one product |
| an LLM backend (backend ID) | `foundry-swedencentral-0` | every model of the backend; the APIM trace confirms which backend answered |
| a model or a model alias | `gpt-5.4-mini`, `fast` | the model on every LLM API that can serve it |
| an MCP server | `weather-mcp` | MCP `initialize` |

Names are matched case-insensitively, and a unique part of a name is enough (`chatbot`). Leave `TARGET`
empty to only print the catalogue.

**How the call is chosen.** The backend contract (`GET /version/backend-contract`, read here from its policy
fragment) gives each backend's type and models. The model kind (chat, embeddings, image) and the backend
type decide the request:

| Backend type | Chat | Embeddings / images |
|---|---|---|
| `ai-foundry`, `azure-openai` | universal (`/models/chat/completions`), Azure OpenAI (`/openai/deployments/{model}/…`), unified (`/v1/chat/completions`) | embeddings on the same APIs; images on unified `/v1/images/generations` |
| `external`, `aws-bedrock-mantle`, `gemini-openai` | universal, unified `/v1` (OpenAI format) | embeddings: `external` only |
| `aws-bedrock` | unified `/bedrock/model/{id}/converse` (Converse format) | |
| `gemini` | unified `/gemini/v1beta/models/{id}:generateContent` | |
| `anthropic` | unified `/claude/v1/messages` (Messages format) | |
| `azure-flux`, `azure-mai` | | unified `/v1/images/generations` |

Only the APIs the key may call are used. Realtime, audio and video models are listed but not called.

**Which key.**
- **Contracts and products:** the contract's subscription key. With `KEY_SOURCE = "auto"` it is read from
  the contract's Key Vault secret (what the consuming team uses), falling back to APIM. When the product
  requires a JWT, the Entra test client's token is added (`task entra-setup`).
- **Backends, models, aliases and MCP servers:** APIM's built-in all-access subscription, so no product
  policy is in the way. The unified AI API needs a product, so for unified-only calls (and when the
  all-access subscription is disabled) the notebook uses a product that grants the model instead.

Keys stay in memory: they are never printed.

| Result | Meaning |
|---|---|
| ✅ | the expected status (200 with a valid answer, or the 401 / 403 of a guardrail check) |
| ❌ | another status or an unexpected response; the result column has the error |
| ⏭️ | not called; the result column says why (no API can serve the model, JWT without a test client, …) |

**Pre-requisites:** `az login` with read access to APIM (and to the contract Key Vault for `KEY_SOURCE =
"key-vault"`), `pip install -r requirements.txt`, and `environments/<ENV>/common.tfvars`.

## ⚙️ Settings

Only `ENV` and `TARGET` are needed.

In [ ]:
ENV = "dev"                 # environments/<ENV>
TARGET = ""                 # contract, product, backend, model, alias or MCP server name (or a list of names); "" = catalogue only

PROMPT = "Reply with one short sentence: which model are you?"
INCLUDE = ("chat", "embeddings", "image")   # model kinds to call; "image" generates a real image (slower, billed)
GUARDRAILS = True           # contracts / products: a call without a key must get 401, a model outside allowed_models 403
TRACE = None                # ask APIM which backend answered (debug trace); None = for backend targets only
KEY_SOURCE = "auto"         # contracts: "auto" (contract Key Vault secret, else APIM) | "key-vault" | "apim"
BACKEND_KEY = "auto"        # backends / models / aliases / MCP: "auto" (all-access subscription, else a granting product) | "<product id>"

## 🔎 Catalogue

Everything `TARGET` can name on this environment: the access contracts (APIM products), the LLM backends of
the backend contract, their models and aliases, and the MCP servers.

In [ ]:
import sys, json, importlib
sys.path.insert(1, "shared")
import acceptance as A
import explorer as X
importlib.reload(A); importlib.reload(X)

ex = X.Explorer(ENV, prompt=PROMPT, key_source=KEY_SOURCE, backend_key=BACKEND_KEY)
print(f"Gateway  {ex.gw.gateway_url}  ({ex.gw.apim_name}, {ex.gw.sku}); signed in as {ex.gw.signed_in}")
print(f"Backend contract {ex.contract.get('backendContractVersion', '?')}, release {ex.contract.get('masterVersion', '?')}")
if ex.contract_note:
    print(f"{X.INFO} {ex.contract_note}")

for title, rows in ex.catalogue().items():
    print(f"\n{title.upper()}")
    A.print_table(rows) if rows else print("   none")

## 🧪 Test

Resolves `TARGET`, prints what it is and the calls it plans, then runs them. A failing call never stops the
cell. Guardrail rows (`no-key`, `denied-model`, `missing-role`) pass when the gateway **rejects** the call.

In [ ]:
targets = [TARGET] if isinstance(TARGET, str) else list(TARGET)
results = []
for target in [t for t in targets if t]:
    try:
        description, steps = ex.plan(target, include=INCLUDE, guardrails=GUARDRAILS)
    except ValueError as e:
        print(f"{X.FAIL} {e}")
        continue
    trace = TRACE if TRACE is not None else description.startswith("backend ")
    print(f"\n▶ {target}: {description}; {len([s for s in steps if s.url])} call(s){' with APIM trace' if trace else ''}")
    for product in sorted({s.product for s in steps if s.url and s.send_key}):
        print(f"   key of {product}: {ex.key(product)[1]}")
    run = ex.run(steps, trace=trace)
    results += run
    A.print_table(X.rows(run))
    print(f"   {X.summary(run)}")
if not any(targets):
    print("TARGET is empty: pick a name from the catalogue above")

## 🔬 Details (optional)

The request and response of one row of the last test (index from 0). The key is never shown.

In [ ]:
DETAIL = None   # e.g. 0 = first row

if DETAIL is not None and results:
    r = results[DETAIL]
    s = r.step
    print(f"{r.icon} {s.method} {s.url}")
    print(f"   product {s.product}, API {s.api}, model {s.model} ({s.kind}, {s.backend_type}), expected {s.expect}")
    print(f"   body    {json.dumps(s.body)[:500] if s.body else ''}")
    if r.call:
        print(f"   HTTP {r.call.status} in {r.call.latency_ms} ms")
        for h in ("apim-request-id", "UAIG-Request-Id", "UAIG-Model-Id", "UAIG-Backend", "UAIG-Semantic-Cache", "x-ms-region", "Retry-After"):
            if h in r.call.headers:
                print(f"   {h}: {r.call.headers[h]}")
        print(f"   response {r.call.text[:1500]}")
    else:
        print(f"   not called: {r.detail}")